# Pourquoi un modèle, et comment on sait qu'il tient

Deux questions qu'un comité vous posera, dans cet ordre : **à quoi sert votre modèle par rapport à la règle qu'on applique déjà ?** et **que se passe-t-il quand les données bougent ?** Ce notebook y répond avec les chiffres du dépôt. Tout est calculé sur le pli de validation ; le test reste fermé jusqu'au jalon 2.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from sklearn.metrics import roc_auc_score
from credit.config import load_config
from credit.features import split_xy, make_pipeline
from credit.model import load_model
from credit.split import load_fold
from credit.evaluate import expected_cost, best_threshold, cost_curve

cfg = load_config()
val = load_fold("validation")
X, y = split_xy(val, cfg); y = y.to_numpy()
m = load_model("baseline")
p = m.predict_proba(X)[:, 1]
c_fp, c_fn = cfg["costs"]["c_fp"], cfg["costs"]["c_fn"]

## 1. Le modèle contre les règles qu'une banque applique déjà

Avant tout machine learning, une banque refuse les clients déjà en retard. Trois règles de ce type, le modèle naïf (tout accepter, tout refuser), et la logistique. La question n'est pas « qui a la meilleure AUC », c'est « qui coûte le moins en euros par dossier ».

In [ ]:
rows = []
for k in (1, 2, 3):
    r = (val["PAY_0"] >= k).astype(float).to_numpy()
    rows.append((f"refuser si PAY_0 >= {k}", r.mean(), roc_auc_score(y, r), expected_cost(y, r, 0.5, c_fp, c_fn)))
rows.append(("accepter tout le monde", 0.0, 0.5, expected_cost(y, np.zeros(len(y)), 0.5, c_fp, c_fn)))
rows.append(("refuser tout le monde", 1.0, 0.5, expected_cost(y, np.ones(len(y)), 0.5, c_fp, c_fn)))
t = cfg["threshold"]
rows.append((f"logistique, seuil {t}", (p >= t).mean(), roc_auc_score(y, p), expected_cost(y, p, t, c_fp, c_fn)))
pd.DataFrame(rows, columns=["décision", "part refusée", "AUC", "coût par dossier (€)"]).round(3)

Lecture : avec un défaut accepté à 4 500 € et un refus à tort à 600 €, refuser presque tout le monde coûte 467 € par dossier, et la logistique 469 €. **À ces coûts, le modèle de référence ne vaut pas mieux que le pire des bons sens.** Ce n'est pas un échec du machine learning, c'est un fait sur ces coûts et ce modèle : l'AUC de 0,72 ne trie pas assez les bas risques. D'où le jalon 2.

Mais les coûts sont des hypothèses. Regardons ce qui se passe quand ils changent.

In [ ]:
rows = []
rule = (val["PAY_0"] >= 1).astype(float).to_numpy()
for cfn in (600, 1000, 1500, 2000, 3000, 4500):
    tt = best_threshold(y, p, c_fp, cfn)
    rows.append((cfn, expected_cost(y, np.ones(len(y)), 0.5, c_fp, cfn), expected_cost(y, np.zeros(len(y)), 0.5, c_fp, cfn),
                 expected_cost(y, rule, 0.5, c_fp, cfn), expected_cost(y, p, tt, c_fp, cfn), tt))
pd.DataFrame(rows, columns=["coût d'un défaut accepté", "refuser tous", "accepter tous", "règle PAY_0 >= 1", "logistique au meilleur seuil", "seuil"]).round(1)

Lecture : à chaque niveau de coût, la logistique fait au moins aussi bien que la meilleure règle, et nettement mieux quand les deux erreurs coûtent à peu près pareil (111 € contre 133 € par dossier à 600 € : 16 % de moins, soit 130 000 € par an pour 6 000 dossiers). **L'intérêt du modèle dépend des coûts**, et c'est au comité de les donner. Le vôtre, au jalon 2, devra battre la logistique de la même façon, chiffres en main.

## 2. Un intervalle, pas un chiffre

« AUC 0,723 » ne veut rien dire seul. Sur 1 000 rééchantillons de la validation, voici où elle tombe.

In [ ]:
rng = np.random.default_rng(0)
aucs = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))]
print("AUC", round(roc_auc_score(y, p), 3), " IC à 95 % :", np.round(np.percentile(aucs, [2.5, 97.5]), 3))

Un modèle qui annoncerait 0,735 ne serait pas meilleur que celui-ci : l'écart est dans l'intervalle. C'est la règle de la séance 6 : pas de champion sans intervalle.

## 3. Les tests de robustesse, un par un

Chacun est aussi un test automatique dans `tests/test_robustness.py`. On les écrit avant de regarder les chiffres : ce sont des propriétés attendues, pas des constats.

In [ ]:
# (a) Du bruit de 1 % sur les montants : rien ne doit bouger de façon visible.
rng = np.random.default_rng(1); Xn = X.copy()
for c in [c for c in X.columns if "AMT" in c or c == "LIMIT_BAL"]:
    Xn[c] = Xn[c] * (1 + rng.normal(0, 0.01, len(Xn)))
pn = m.predict_proba(Xn)[:, 1]
print("écart max sur p :", round(np.abs(pn - p).max(), 4), " AUC :", round(roc_auc_score(y, pn), 4))

In [ ]:
# (b) Monotonie : un mois de retard de plus doit augmenter le risque, pour tout le monde.
X2 = X.copy(); X2["PAY_0"] = X2["PAY_0"] + 1
p2 = m.predict_proba(X2)[:, 1]
print("part des clients dont le risque monte :", (p2 > p).mean(), " hausse moyenne :", round((p2 - p).mean(), 3))

In [ ]:
# (c) Valeurs absurdes : le modèle ne plante pas et répond une probabilité.
Xe = X.iloc[:3].copy(); Xe["LIMIT_BAL"] = 1e9; Xe["AGE"] = 200
print(m.predict_proba(Xe)[:, 1])
# Attention : 0,0 n'est pas une bonne réponse pour un client absurde. En production, un contrat de
# schéma (bornes) doit rejeter la ligne AVANT le modèle. Voir credit.data.check_schema.

In [ ]:
# (d) Stabilité : si le pli d'entraînement avait été tiré autrement, aurait-on le même modèle ?
from sklearn.linear_model import LogisticRegression
Xt, yt = split_xy(load_fold("train"), cfg); res = []
for s in range(5):
    i = np.random.default_rng(s).integers(0, len(Xt), len(Xt))
    pipe = make_pipeline(LogisticRegression(max_iter=2000), cfg).fit(Xt.iloc[i], yt.iloc[i])
    res.append(roc_auc_score(y, pipe.predict_proba(X)[:, 1]))
print("AUC sur cinq rééchantillons de l'entraînement :", np.round(res, 4))

In [ ]:
# (e) Tranches : le modèle marche-t-il aussi bien pour chaque groupe ? (prélude à l'audit d'équité du jalon 2)
rows = []
for g, grp in val.groupby("MARRIAGE"):
    if len(grp) >= 500:
        pg = m.predict_proba(grp[X.columns])[:, 1]
        rows.append((g, len(grp), grp["default"].mean(), roc_auc_score(grp["default"], pg), (pg >= t).mean()))
pd.DataFrame(rows, columns=["MARRIAGE", "n", "taux de défaut", "AUC", "part refusée"]).round(3)

Même AUC dans les deux groupes, mais une part refusée différente, qui suit le taux de défaut observé. Est-ce acceptable ? C'est une question d'équité, pas de performance, et elle se traite en séance 7.

In [ ]:
# (f) Dérive : si demain tous les plafonds doublaient, que dirait le modèle ?
X3 = X.copy(); X3["LIMIT_BAL"] = X3["LIMIT_BAL"] * 2
print("risque moyen prédit :", round(p.mean(), 4), "→", round(m.predict_proba(X3)[:, 1].mean(), 4))
# Le modèle baisse le risque de 1,4 point. Vrai ou faux ? Personne ne le sait : il n'a jamais vu ces
# clients. C'est pour cela qu'on surveille la dérive des entrées (PSI, séance 8) et qu'on réentraîne.

## À retenir

1. L'intérêt d'un modèle se mesure en euros contre la règle en place, et il dépend des coûts d'erreur : écrivez-les.
2. Un chiffre sans intervalle n'est pas un résultat.
3. Un modèle robuste est un modèle dont on a écrit, avant de regarder, ce qu'il doit faire quand les données bougent : bruit, monotonie, valeurs absurdes, rééchantillonnage, tranches, dérive.
4. Les tests de `tests/test_robustness.py` tournent à chaque pull request : si votre modèle lourd du jalon 2 les casse, c'est lui qui a tort.